# SAADE v4 — перегляд аудіо та результатів

Відкривайте **після виконання основного експерименту**. Цей ноутбук не навчає моделі й не змінює AUA.


In [ ]:
from pathlib import Path
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from IPython.display import display
import saade_v3_core as S
import saade_v3_visuals as V
P=Path('/home/jovyan/work/audio_real_fake')
OUT=P/'results'/'SAADE_AUA_2026_v4_ADAPTIVE_THESIS'
CFG=S.Config(root=str(P/'AUA'),out=str(OUT),scenario_profile='v2_5level')
assert OUT.is_dir(),str(OUT)
print('Відкрито результати:',OUT)


## 1. Збалансовані вибірки та ризики незалежності

In [ ]:
for name in ['V4_02_BALANCE_REPORT.csv','V4_02_FAKE_GROUP_RISKS.csv','V4_02_SPLIT_SOURCE_CLASS.csv']:
  p=OUT/name
  if p.is_file():
    print(name);display(pd.read_csv(p).head(30))


## 2. Аудіоплеєри: усі 21 сценарій, хвильові форми і спектрограми

In [ ]:
p=OUT/'V4_02_LOCKED_EXPERIMENT_SEGMENTS.csv'
assert p.is_file(),f'Не знайдено: {p}'
splits=pd.read_csv(p)
from IPython.display import Audio
for record in V.pick_examples(splits,'validation'):
    print('REAL' if int(record.label)==0 else 'FAKE',record.relative_path)
    V.show_audio_gallery(record,CFG,tag='v4_viewer')
V.make_interactive_player(splits,CFG,'validation')

## 3. E1/E2/E3 за сценаріями та seeds

In [ ]:
p=OUT/'V4_05_SCENARIO_METRICS.csv'
if p.is_file():
  z=pd.read_csv(p)
  display(z.head(25))
  for metric in ('balanced_accuracy','eer','roc_auc','f1_fake'):
    d=z[z.category!='baseline'].groupby(['model','regime','category','level'])[metric].mean().reset_index()
    for model,df in d.groupby('model'):
      fig,ax=plt.subplots(figsize=(10,4))
      for regime,r in df.groupby('regime'):
        x=r.groupby('level')[metric].mean().sort_index()
        ax.plot(x.index,x.values,marker='o',label=regime)
      ax.set(title=f'{model}: {metric} середнє за типами сценаріїв',xlabel='Рівень спотворення',ylabel=metric)
      ax.legend();ax.grid(alpha=.2);plt.show();plt.close(fig)
else:print('Тестування ще не запускалося.')

## 4. Чи виграла адаптивність? Без змішування seed із кількістю аудіозаписів

In [ ]:
for filename in ['V4_05_PAIRED_SEED_DELTAS.csv','V4_05_DELTA_SUMMARY.csv','V4_07_RESEARCH_READINESS.json']:
  p=OUT/filename
  if p.is_file():
    print(filename)
    if p.suffix=='.json': print(p.read_text(encoding='utf-8'))
    else: display(pd.read_csv(p))